# 同位素质量平衡模型 - 基础使用示例

本 Notebook 展示如何使用框架的核心功能：

1. 核心数学工具（ODE求解、插值）
2. 同位素公式（Delta计算、混合、瑞利分馏）
3. 各同位素体系（Mg、C、N、U）

## 环境准备

In [1]:
# 添加项目根目录到路径
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent.parent))

import numpy as np
import pandas as pd

print("✓ 环境准备完成")

✓ 环境准备完成


---

## 1. 核心数学工具

### 1.1 ODE 求解器

求解指数衰减方程：dy/dt = -k*y

In [2]:
from toolkit.math.numerical import ODESolver

# 定义ODE: 指数衰减
def exponential_decay(t, y, k):
    return -k * y

# 求解
result = ODESolver.solve(
    func=exponential_decay,
    y0=1.0,                    # 初始条件
    t_span=(0, 10),            # 时间范围
    args=(0.5,),               # k = 0.5
    method='RK45',
    n_points=100
)

if result.success:
    y_arr = np.array(result.y)
    if y_arr.ndim > 1:
        y_arr = y_arr.flatten()
    print(f"ODE求解成功: {len(result.t)} 个点")
    print(f"  y(0) = {y_arr[0]:.4f}")
    print(f"  y(10) = {y_arr[-1]:.4f}")
    print(f"  理论值: {np.exp(-0.5 * 10):.4f}")

ODE求解成功: 100 个点
  y(0) = 1.0000
  y(10) = 0.0068
  理论值: 0.0067


### 1.2 插值工具

In [3]:
from toolkit.math.numerical import Interpolator

# 原始数据: y = x^2
x = np.array([0, 1, 2, 3, 4])
y = np.array([0, 1, 4, 9, 16])

# 插值到更密集的网格
x_new = np.linspace(0, 4, 20)
y_interp = Interpolator.interpolate(x, y, x_new, method='cubic')

print(f"插值: {len(x)} 个点 → {len(x_new)} 个点")
print(f"x = 2 时, y = {Interpolator.interpolate(x, y, [2.0], method='cubic')[0]:.2f} (理论: 4.0)")

插值: 5 个点 → 20 个点
x = 2 时, y = 4.00 (理论: 4.0)


---

## 2. 同位素公式

### 2.1 Delta 值计算

In [4]:
from toolkit.isotope.formulas import DeltaCalculator

# VPDB 标准
standard_ratio = 0.0112372
sample_ratio = 0.0111500

# 计算 delta13C
delta = DeltaCalculator.ratio_to_delta(sample_ratio, standard_ratio)
print(f"样品比值: {sample_ratio}")
print(f"δ¹³C: {delta:.2f}‰")

样品比值: 0.01115
δ¹³C: -7.76‰


### 2.2 混合计算

In [5]:
from toolkit.isotope.formulas import MassBalance

# 碳酸盐 vs 有机碳混合
delta_carb = -4.0    # 碳酸盐
delta_org = -30.0    # 有机碳
f_carb = 0.7         # 碳酸盐比例

delta_mix = MassBalance.two_component_mixing(delta_carb, delta_org, f_carb)

print(f"端元A (碳酸盐): {delta_carb}‰ ({f_carb*100:.0f}%)")
print(f"端元B (有机碳): {delta_org}‰ ({(1-f_carb)*100:.0f}%)")
print(f"混合结果: {delta_mix:.2f}‰")

端元A (碳酸盐): -4.0‰ (70%)
端元B (有机碳): -30.0‰ (30%)
混合结果: -11.80‰


### 2.3 瑞利分馏

In [6]:
from toolkit.isotope.formulas import RayleighFractionation

# 瑞利分馏: 残余相
f_residual = 0.5     # 50% 剩余
alpha = 1.025        # 分馏系数 (+25‰)

delta_residual = RayleighFractionation.residual_fraction(f_residual, alpha)

print(f"残余比例: {f_residual}")
print(f"残余相 δ: {delta_residual:.2f}‰")

残余比例: 0.5
残余相 δ: -17.18‰


---

## 3. 同位素体系

### 3.1 Mg 同位素 - 风化分析

In [7]:
from systems.mg import MgIsotopeSystem

# 创建体系实例
mg_system = MgIsotopeSystem()

# 风化比例计算
delta_sample = -2.5  # 样品值
delta_sw = -0.83     # 海水值

ratios = mg_system.calculate_weathering_ratio(delta_sample, delta_sw)

print(f"样品 δ²⁶Mg: {delta_sample}‰")
print(f"海水 δ²⁶Mg: {delta_sw}‰")
print(f"碳酸盐风化比例: {ratios['f_carbonate']:.1%}")
print(f"硅酸盐风化比例: {ratios['f_silicate']:.1%}")

样品 δ²⁶Mg: -2.5‰
海水 δ²⁶Mg: -0.83‰
碳酸盐风化比例: 0.0%
硅酸盐风化比例: 100.0%


### 3.2 C 同位素 - DOC 模型

In [8]:
from systems.c import CIsotopeSystem

# DICE情景
c_system = CIsotopeSystem(scenario='dice')

# 计算特定DOC通量下的稳态
F_odoc = 4.0e18  # mol/Ma
result = c_system.solve_steady_state(F_odoc=F_odoc)

if result.success:
    print(f"DOC通量: {F_odoc:.2e} mol/Ma")
    print(f"δ¹³C_carb: {result.get('delta13C_carb'):.2f}‰")
    print(f"δ¹³C_org: {result.get('delta13C_org'):.2f}‰")
    
    # 碳同位素漂移
    initial = -4.0  # 初始海水DIC
    excursion = result.get('delta13C_carb') - initial
    print(f"碳同位素漂移: {excursion:.2f}‰")

DOC通量: 4.00e+18 mol/Ma
δ¹³C_carb: -3.39‰
δ¹³C_org: -33.39‰
碳同位素漂移: 0.61‰


### 3.3 N 同位素 - 硝酸盐可利用性

In [9]:
from systems.n import NIsotopeSystem

# 早三叠世情景
n_system = NIsotopeSystem(scenario='early_triassic')

# 正向模型: f_assimilator → δ¹⁵N
f_values = [0.0, 0.11, 0.25, 0.48, 0.7]

print(f"{'f_assimilator':<15} {'δ¹⁵N_sed':<12} {'解释'}")
print("-" * 50)

for f in f_values:
    delta15N = n_system.forward_model(f_assimilator=f)
    if f < 0.1:
        status = "极度缺氧"
    elif f < 0.2:
        status = "硝酸盐受限"
    elif f < 0.4:
        status = "中等可利用性"
    else:
        status = "充足"
    print(f"{f:<15.2f} {delta15N:<+12.2f} {status}")

f_assimilator   δ¹⁵N_sed     解释
--------------------------------------------------
0.00            -0.50        极度缺氧
0.11            +2.11        硝酸盐受限
0.25            +4.49        中等可利用性
0.48            +6.15        充足
0.70            +5.09        充足


### 3.4 N 同位素 - 反向反演

In [10]:
# 从观测的 δ¹⁵N 反演 f_assimilator
observed_delta15N = 3.0

result = n_system.inverse_model(
    delta15N_sed=observed_delta15N,
    f_range=(0.0, 0.48)  # 早三叠世合理范围
)

print(f"观测 δ¹⁵N: {observed_delta15N:.2f}‰")
print(f"反演 f_assimilator: {result['f_assimilator']:.3f}")
print(f"残差: {result['residual']:+.4f}‰")

观测 δ¹⁵N: 3.00‰
反演 f_assimilator: 0.156
残差: +0.0000‰


### 3.5 U 同位素 - 缺氧比例计算

In [ ]:
from systems.u import UIsotopeSystem

u_system = UIsotopeSystem(scenario='modern')

# 从碳酸盐 δ238U 计算缺氧比例
delta238_carb = -0.65

result = u_system.calculate_f_anox_steady_state(delta238_carb)

print(f"碳酸盐 δ238U: {delta238_carb:.2f}‰")
print(f"海水 δ238U: {result['delta238_seawater']:+.2f}‰")
print(f"缺氧汇比例 f_anox: {result['f_anox']:.1%}")
print(f"氧化汇比例 f_oxic: {result['f_oxic']:.1%}")

# 估算缺氧面积
anoxic_area = u_system.estimate_anoxic_area(result['f_anox'])
print(f"估算缺氧海底面积: ~{anoxic_area:.1f}%")

碳酸盐 δ²³⁸U: -0.65‰
海水 δ²³⁸U: -1.05‰
缺氧汇比例 f_anox: 98.7%
氧化汇比例 f_oxic: 1.3%
估算缺氧海底面积: ~2.5%


---

## 4. 批量计算与输出

生成 N 同位素关系曲线并保存

In [12]:
# 计算关系曲线
curve = n_system.calculate_f_assimilator_curve(
    f_range=(0.0, 1.0),
    n_points=50,
    n_monte_carlo=1000
)

# 保存到CSV
df = pd.DataFrame({
    'f_assimilator': curve['f_assimilator'],
    'delta15N_mean': curve['delta15N_sed_mean'],
    'delta15N_ci68_lower': curve['delta15N_sed_ci68_lower'],
    'delta15N_ci68_upper': curve['delta15N_sed_ci68_upper']
})

output_path = 'n_isotope_curve.csv'
df.to_csv(output_path, index=False)
print(f"✓ 结果已保存到: {output_path}")

# 显示前5行
df.head()

✓ 结果已保存到: n_isotope_curve.csv


,f_assimilator,delta15N_mean,delta15N_ci68_lower,delta15N_ci68_upper
0,0.000000,-0.471856,-1.498839,0.501688
1,0.020408,0.036945,-1.002412,1.012738
2,0.040816,0.544618,-0.504452,1.560683
3,0.061224,1.014865,0.061638,1.983172
4,0.081633,1.489780,0.498727,2.496822


---

## 总结

本 Notebook 展示了：

1. **核心工具**: ODE求解、插值
2. **同位素公式**: Delta转换、混合、瑞利分馏
3. **各同位素体系**: Mg、C、N、U的基本用法
4. **数据处理**: 批量计算与CSV输出

更多示例请参考：
- CLI用法: `python cli.py --help`
- ODE求解器文档: `docs/ODESOLVER_GUIDE.md`
- 项目架构: `ARCHITECTURE.md`